The purpose of this notebook is to test the ingestion process from EIA API  

In [ ]:
import requests
import pandas as pd
from datetime import datetime

import os
from dotenv import load_dotenv

from sqlalchemy import (
    Column, Table, MetaData, text, Text, Numeric, Integer, BigInteger, Boolean, DataTime, create_engine, inspect
)

from sqlalchemy.dialects.postgresql import insert as pg_insert

In [ ]:
TYPE_MAP = {
    'TEXT' : Text,
    'NUMERIC' : Numeric,
    'INTEGER' : Integer,
    'BIGINT' : BigInteger,
    'BOOLEAN' : Boolean,
    'TIMESTAMP' : DataTime,
}

In [2]:
API_KEY = os.getenv("EIA_API_KEY")

# Consider adding a try-except block to handle potential errors in the API request and missing API keys

In [4]:
BASE_URL = "https://api.eia.gov/v2"

In [5]:
def eia_fetch(route: str, params: dict) -> pd.DataFrame:
    """
    Fetch data from the EIA API and return it as a pandas DataFrame.

    Parameters:
    - route: The API endpoint route (e.g., 'series')
    - params: A dictionary of parameters to include in the API request

    Returns:
    - A pandas DataFrame containing the fetched data
    """
    url = f"{BASE_URL}/{route}/data/"
    
    # Add the API key to the parameters
    params['api_key'] = API_KEY
    params["offset"] = params.get("offset", 0) 
    params["length"] = params.get("length", 5000) # max per request
    
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()  # Raise an error for bad status codes
        
        payload = response.json()['response']      
        # Assuming the data is in a specific format, you may need to adjust this based on the actual response structure
        df = pd.DataFrame(payload['data'])
        return df
    except requests.exceptions.RequestException as e:
        print(f"An error occurred while fetching data: {e}")
        return pd.DataFrame()  # Return an empty DataFrame in case of error

In [ ]:
def eia_fetch_all(route: str, params: dict) -> pd.DataFrame:
    """
    Fetch data from the EIA API and return it as a pandas DataFrame.
    Modified to handle pagination and fetch all available data.

    Parameters:
    - route: The API endpoint route (e.g., 'series')
    - params: A dictionary of parameters to include in the API request

    Returns:
    - A pandas DataFrame containing the fetched data
    """

    all_records = []
    offset = 0
    page_size = 5000 # max per request from eia.gov
    
    while True: 

        params['offset'] = offset
        params['length'] = page_size
        params['api_key'] = API_KEY

        url = f"{BASE_URL}/{route}/data/"
    
        try:
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()  # Raise an error for bad status codes
            
            payload = response.json()['response']      
            records = payload['data']
            total = int(payload.get("total", 0))

            all_records.extend(records)
            offset += page_size

            if offset >= total:
                break  # Exit the loop if we've fetched all records
            
        except requests.exceptions.RequestException as e:
            print(f"An error occurred while fetching data: {e}")
            return pd.DataFrame()  # Return an empty DataFrame in case of error
        
    return pd.DataFrame(all_records)

In [ ]:
def create_postgres_engine(user: str, password: str, host: str, port: int, database: str) -> create_engine:
    """
    Create a SQLAlchemy engine for connecting to a PostgreSQL database.

    Parameters:
    - user: Database username
    - password: Database password
    - host: Database host (e.g., 'localhost')
    - port: Database port (e.g., 5432)
    - database: Database name

    Returns:
    - A SQLAlchemy engine instance
    """
    connection_string = f"postgresql://{user}:{password}@{host}:{port}/{database}"
    engine = create_engine(connection_string)
    return engine

In [ ]:
def write_to_postgres(
        df: pd.DataFrame, 
        table_name: str, 
        schema: dict, 
        engine: create_engine,
        pg_schema: str = "raw"
        ) -> None: 
    """
    Dynamically create or upsert a PostgreSQL table from a DataFrame.

    Schema format:
        { "column_name": ("DATA_TYPE", is_primary_key) }

    Behaviour:
        - Table doesn't exist  → creates it with defined PKs, then inserts
        - Table exists         → upserts on conflict of PK columns
    """

    metadata = MetaData(schema=pg_schema)
    inspector = inspect(engine)

    # --- Build Column objects from schema dict --- 
    columns = []
    pk_columns = [] 

    for col_name, (dtype, is_pk) in schema.items():
        col_type = TYPE_MAP.get(dtype.upper())
        if not col_type:
            raise ValueError(f"Unsupported data type: {dtype} for column: {col_name}")
        
        columns.append(Column(col_name, col_type, primary_key=is_pk))
        if is_pk:
            pk_columns.append(col_name)

    if not pk_columns:
        raise ValueError("At least one primary key column must be defined in the schema.")

    # --- Define the table object --- 
    table = Table(table_name, metadata, *columns)

    # --- Create table if it doesn't exist --- 
    existing_tables = inspector.get_table_names(schema=pg_schema)
    if table_name not in existing_tables:
        metadata.create_all(engine)
        print(f"Table '{pg_schema}.{table_name}' created successfully."
              f" with Primary Key(s): {pk_columns}")

    # --- Only keep columns that exist in the schema --- 
    df = df[[col for col in schema.keys() if col in df.columns]].copy()

    # --- Upsert in batches --- 
    records = df.to_dict(orient='records')
    non_pk_cols = [c for c in schema.keys() if c not in pk_cols and c in df.columns]

    with engine.begin() as conn: # auto-commits or rolls back on exception 
        
        stmt = pg_insert(table).values(records) # Statement for bulk insert

        if non_pk_cols:
            # Update all non-PK columns on conflict
            upsert_stmt = stmt.on_conflict_do_update(
                index_elements=pk_columns,
                set={col: stmt.excluded[col] for col in non_pk_cols}
            )
        else:
            # All columns are PKs - just do nothing on conflict
            upsert_stmt = stmt.on_conflict_do_nothing(index_elements=pk_columns)

        conn.execute(upsert_stmt)
    
    print(f"[INFO] Upserted {len(records)} records into '{pg_schema}.{table_name}' successfully.")






SyntaxError: invalid syntax. Perhaps you forgot a comma? (3457277963.py, line 25)

In [23]:
# --- Crude Oil Production (monthly, by state) ---
params = {
    "frequency": "monthly",
    "data[0]": "value",
    "facets[series][]": "MCRFPTX2",   # Texas field production of crude oil
    "start": "2015-01",
    "end": "2024-12",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
}
df_production = eia_fetch_all("petroleum/crd/crpdn", params)
display(df_production)

120


,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
0,2015-01,STX,TEXAS,EPC0,Crude Oil,FPF,Field Production,MCRFPTX2,Texas Field Production of Crude Oil (Thousand ...,3442,MBBL/D
1,2015-02,STX,TEXAS,EPC0,Crude Oil,FPF,Field Production,MCRFPTX2,Texas Field Production of Crude Oil (Thousand ...,3557,MBBL/D
2,2015-03,STX,TEXAS,EPC0,Crude Oil,FPF,Field Production,MCRFPTX2,Texas Field Production of Crude Oil (Thousand ...,3619,MBBL/D
3,2015-04,STX,TEXAS,EPC0,Crude Oil,FPF,Field Production,MCRFPTX2,Texas Field Production of Crude Oil (Thousand ...,3571,MBBL/D
4,2015-05,STX,TEXAS,EPC0,Crude Oil,FPF,Field Production,MCRFPTX2,Texas Field Production of Crude Oil (Thousand ...,3511,MBBL/D
...,...,...,...,...,...,...,...,...,...,...,...
115,2024-08,STX,TEXAS,EPC0,Crude Oil,FPF,Field Production,MCRFPTX2,Texas Field Production of Crude Oil (Thousand ...,5805,MBBL/D
116,2024-09,STX,TEXAS,EPC0,Crude Oil,FPF,Field Production,MCRFPTX2,Texas Field Production of Crude Oil (Thousand ...,5798,MBBL/D
117,2024-10,STX,TEXAS,EPC0,Crude Oil,FPF,Field Production,MCRFPTX2,Texas Field Production of Crude Oil (Thousand ...,5832,MBBL/D
118,2024-11,STX,TEXAS,EPC0,Crude Oil,FPF,Field Production,MCRFPTX2,Texas Field Production of Crude Oil (Thousand ...,5768,MBBL/D


In [ ]:
# --- WTI Crude Spot Price (daily) ---
params = {
    "frequency": "daily",
    "data[0]": "value",
    "facets[series][]": "RWTC",
    "start": "2015-01-01",
    "end": "2024-12-31",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
}
df_wti = eia_fetch("petroleum/pri/spt", params)
display(df_wti)

In [ ]:
# --- Natural Gas Spot Price (Henry Hub, daily) ---
params = {
    "frequency": "daily",
    "data[0]": "value",
    "facets[series][]": "RNGWHHD",
    "start": "2015-01-01",
    "end": "2024-12-31",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
}
df_ng_price = eia_fetch("natural-gas/pri/sum", params)
display(df_ng_price)

In [ ]:
# --- Weekly Crude Inventory (US total stocks) ---
params = {
    "frequency": "weekly",
    "data[0]": "value",
    "facets[series][]": "WTTSTUS1",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
}
df_inventory = eia_fetch("petroleum/stoc/wstk", params)
display(df_inventory)